# Assignment Beta – Question 3: Knowledge Distillation
**Syed Ukkashah [23K-0055]**

Prompt: *"The patient was admitted to the ___"*

| Index | Word |
|---|---|
| 1 | hospital |
| 2 | clinic |
| 3 | ward |
| 4 | school |

Correct word = **hospital**, so the one-hot label is $y = [1, 0, 0, 0]$.

Loss: $L = \alpha \cdot T^2 \cdot KL(p_T \,\|\, q_T) + (1-\alpha)\cdot CE(y, q_1)$ with $T = 2$, $\alpha = 0.5$. Natural log throughout.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)

words = ["hospital", "clinic", "ward", "school"]
y   = np.array([1.0, 0.0, 0.0, 0.0])      # one-hot label
z_t = np.array([5.0, 3.0, 2.0, 0.0])      # teacher logits
z_s = np.array([3.0, 2.5, 1.0, 0.5])      # student logits
T, alpha, lr = 2.0, 0.5, 0.5

## (a) Probabilities at T = 1

$\text{softmax}(z, T)_i = \dfrac{e^{z_i/T}}{\sum_j e^{z_j/T}}$

The helper below also prints the intermediate values ($z/T$, $e^{z/T}$ and the sum) so every step of the working is visible.

In [2]:
def softmax(z, T=1.0, show=False, name=""):
    scaled = z / T
    exps = np.exp(scaled)
    probs = exps / exps.sum()
    if show:
        print(f"{name}:  z/T = {scaled}")
        print(f"{' ' * len(name)}   e^(z/T) = {exps}   sum = {exps.sum():.4f}")
        print(f"{' ' * len(name)}   p = {probs}\n")
    return probs

p1 = softmax(z_t, 1, show=True, name="Teacher T=1")
q1 = softmax(z_s, 1, show=True, name="Student T=1")

Teacher T=1:  z/T = [5. 3. 2. 0.]
              e^(z/T) = [148.4132  20.0855   7.3891   1.    ]   sum = 176.8878
              p = [0.839  0.1135 0.0418 0.0057]

Student T=1:  z/T = [3.  2.5 1.  0.5]
              e^(z/T) = [20.0855 12.1825  2.7183  1.6487]   sum = 36.6350
              p = [0.5483 0.3325 0.0742 0.045 ]



## (b) Softened probabilities at T = 2

Dividing the logits by T = 2 flattens both distributions, which exposes the teacher's "dark knowledge": *clinic* and *ward* are much more plausible than *school*.

In [3]:
p_T = softmax(z_t, T, show=True, name="Teacher T=2")
q_T = softmax(z_s, T, show=True, name="Student T=2")

Teacher T=2:  z/T = [2.5 1.5 1.  0. ]
              e^(z/T) = [12.1825  4.4817  2.7183  1.    ]   sum = 20.3825
              p = [0.5977 0.2199 0.1334 0.0491]

Student T=2:  z/T = [1.5  1.25 0.5  0.25]
              e^(z/T) = [4.4817 3.4903 1.6487 1.284 ]   sum = 10.9048
              p = [0.411  0.3201 0.1512 0.1177]



## (c) KL divergence between the softened distributions

$KL(p_T \,\|\, q_T) = \sum_i p_{T,i} \,\ln\dfrac{p_{T,i}}{q_{T,i}}$

In [4]:
def kl_divergence(p, q, show=False):
    terms = p * np.log(p / q)
    if show:
        for w, pi, qi, t in zip(words, p, q, terms):
            print(f"{w:9s}: {pi:.4f} * ln({pi:.4f}/{qi:.4f}) = {t:+.4f}")
        print(f"KL = {terms.sum():.4f}")
    return terms.sum()

kl = kl_divergence(p_T, q_T, show=True)

hospital : 0.5977 * ln(0.5977/0.4110) = +0.2239
clinic   : 0.2199 * ln(0.2199/0.3201) = -0.0826
ward     : 0.1334 * ln(0.1334/0.1512) = -0.0167
school   : 0.0491 * ln(0.0491/0.1177) = -0.0430
KL = 0.0816


## (d) Hard-label cross-entropy

$CE(y, q_1) = -\sum_i y_i \ln q_{1,i} = -\ln q_{1,\text{hospital}}$

In [5]:
def cross_entropy(y, q):
    return -np.sum(y * np.log(q))

ce = cross_entropy(y, q1)
print(f"CE = -ln({q1[0]:.4f}) = {ce:.4f}")

CE = -ln(0.5483) = 0.6010


## (e) Total distillation loss

$L = 0.5 \cdot 2^2 \cdot KL + 0.5 \cdot CE = 2 \cdot KL + 0.5 \cdot CE$

In [6]:
def distill_loss(z_s, show=False):
    q_T = softmax(z_s, T)
    q1 = softmax(z_s, 1)
    soft = alpha * T**2 * kl_divergence(p_T, q_T)
    hard = (1 - alpha) * cross_entropy(y, q1)
    if show:
        print(f"soft part = {alpha} * {T**2:.0f} * {kl_divergence(p_T, q_T):.4f} = {soft:.4f}")
        print(f"hard part = {1 - alpha} * {cross_entropy(y, q1):.4f} = {hard:.4f}")
        print(f"L = {soft:.4f} + {hard:.4f} = {soft + hard:.4f}")
    return soft + hard

L0 = distill_loss(z_s, show=True)

soft part = 0.5 * 4 * 0.0816 = 0.1632
hard part = 0.5 * 0.6010 = 0.3005
L = 0.1632 + 0.3005 = 0.4637


## (f) Gradient with respect to the student's logits

**Derivation.**

* For softmax with cross-entropy, $\dfrac{\partial}{\partial z}\big[-\sum_i y_i \ln \text{softmax}(z)_i\big] = \text{softmax}(z) - y$.
* $KL(p_T\|q_T) = \sum p_T \ln p_T - \sum p_T \ln q_T$. The first term does not depend on $z_s$, and $q_T = \text{softmax}(z_s/T)$. The chain rule adds a factor $1/T$:
  $\dfrac{\partial KL}{\partial z_s} = \dfrac{1}{T}(q_T - p_T)$
* Putting the pieces together:

$$\frac{\partial L}{\partial z_s} = \alpha T^2 \cdot \frac{1}{T}(q_T - p_T) + (1-\alpha)(q_1 - y) = \alpha T (q_T - p_T) + (1-\alpha)(q_1 - y)$$

The $T^2$ factor in the loss is there precisely to cancel the $1/T^2$ shrinkage of soft-target gradients (Hinton et al., 2015).

In [7]:
soft_grad = alpha * T * (q_T - p_T)
hard_grad = (1 - alpha) * (q1 - y)
grad = soft_grad + hard_grad

print("soft part  =", soft_grad)
print("hard part  =", hard_grad)
print("dL/dz_s    =", grad)
print("sum of grad =", round(grad.sum(), 10), "(always 0 for softmax gradients)")

soft part  = [-0.1867  0.1002  0.0178  0.0687]
hard part  = [-0.2259  0.1663  0.0371  0.0225]
dL/dz_s    = [-0.4126  0.2665  0.0549  0.0912]
sum of grad = -0.0 (always 0 for softmax gradients)


**Check:** compare the formula against a numerical (finite-difference) gradient.

In [8]:
eps = 1e-6
numeric = np.array([(distill_loss(z_s + eps * np.eye(4)[i]) - distill_loss(z_s - eps * np.eye(4)[i])) / (2 * eps)
                    for i in range(4)])
print("numerical  =", numeric)
print("max |difference| =", np.abs(numeric - grad).max())

numerical  = [-0.4126  0.2665  0.0549  0.0912]
max |difference| = 1.9252188732110653e-10


## (g) One gradient-descent step (η = 0.5)

$z_s' = z_s - \eta \cdot \nabla_{z_s} L$

In [9]:
z_s_new = z_s - lr * grad
print("old logits =", z_s)
print("new logits =", z_s_new, "\n")

print("New student probabilities:")
q1_new = softmax(z_s_new, 1, show=True, name="Student T=1")
qT_new = softmax(z_s_new, T, show=True, name="Student T=2")

L1 = distill_loss(z_s_new, show=True)
print(f"\nLoss before = {L0:.4f},  after = {L1:.4f},  decrease = {L0 - L1:.4f}")

old logits = [3.  2.5 1.  0.5]
new logits = [3.2063 2.3668 0.9725 0.4544] 

New student probabilities:
Student T=1:  z/T = [3.2063 2.3668 0.9725 0.4544]
              e^(z/T) = [24.6873 10.6629  2.6446  1.5752]   sum = 39.5701
              p = [0.6239 0.2695 0.0668 0.0398]

Student T=2:  z/T = [1.6031 1.1834 0.4863 0.2272]
              e^(z/T) = [4.9686 3.2654 1.6262 1.2551]   sum = 11.1154
              p = [0.447  0.2938 0.1463 0.1129]

soft part = 0.5 * 4 * 0.0567 = 0.1134
hard part = 0.5 * 0.4718 = 0.2359
L = 0.1134 + 0.2359 = 0.3493

Loss before = 0.4637,  after = 0.3493,  decrease = 0.1145


## Conclusion

* The step raises the *hospital* logit and lowers the *clinic* logit. Both the teacher and the label agree that *hospital* should get more probability than the student currently gives it.
* The soft term pushes *school* down about four times harder than *ward* (0.069 vs 0.018). The student gives *school* far more probability than the teacher does (0.118 vs 0.049 at T = 2), while its *ward* estimate is already close (0.151 vs 0.133). The one-hot label treats all wrong words the same, so this "how wrong is each word" information is exactly what distillation adds.
* The total loss goes down after a single step, which confirms the gradient direction is correct. The finite-difference check matches the analytic gradient to better than 1e-9.